# PyTorch Tutorial

Paul J. Atzberger<br> 
<https://web.atzberger.org/>

# Neural Networks with `torch.nn`

PyTorch provides the `torch.nn` module as a high-level building block for
neural networks. It handles:

- **Layers:** `nn.Linear`, `nn.Conv2d`, `nn.BatchNorm1d`, and (others).
- **Activations:** `nn.ReLU`, `nn.Tanh`, `nn.GELU`, and (others).
- **Loss functions:** `nn.MSELoss`, `nn.CrossEntropyLoss`, and (others).
- **Parameter management:** `.parameters()` returns all learnable weights.

In this notebook you will:
1. Understand the `nn.Module` base class
2. Use `nn.Linear` to build a single layer
3. Compose layers with `nn.Sequential`
4. Write a custom `nn.Module` subclass
5. Run a forward pass and inspect the output
6. Explore common loss functions

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

---
## The Building Block: `nn.Linear`

A **linear layer** (also called a *fully-connected* or *dense* layer) computes
$$y = xW^\top + b$$
where $W$ is a weight matrix and $b$ is a bias vector. These both will be learned during training.

The `nn.Linear(in_features, out_features)` creates a layer that maps a vector of
size `in_features` to one of size `out_features`.

In [ ]:
# A single linear layer: 4 inputs → 2 outputs
layer = nn.Linear(in_features=4, out_features=2)

print("Weight shape:", layer.weight.shape)  # (out, in) = (2, 4)
print("Bias shape  :", layer.bias.shape)    # (out,)    = (2,)

# Forward pass: pass a batch of 3 samples, each with 4 features
x = torch.randn(3, 4)      # batch of 3
y = layer(x)               # calls layer.forward(x) internally
print("Input  shape:", x.shape)  # (3, 4)
print("Output shape:", y.shape)  # (3, 2)

---
## Activation Functions

Without activations, stacking linear layers collapses to a single linear
transformation (they compose). Activations introduce non-linearity, giving
the network the ability to learn complex patterns.

In [ ]:
x = torch.linspace(-3, 3, 200)

activations = {
    "ReLU" : nn.ReLU(),
    "Tanh" : nn.Tanh(),
    "Sigmoid": nn.Sigmoid(),
    "GELU" : nn.GELU(),
}

fig, axes = plt.subplots(1, len(activations), figsize=(12, 3))
for ax, (name, fn) in zip(axes, activations.items()):
    y = fn(x).detach()
    ax.plot(x.numpy(), y.numpy())
    ax.axhline(0, color="k", lw=0.5)
    ax.axvline(0, color="k", lw=0.5)
    ax.set_title(name)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Composing Layers with `nn.Sequential`

The `nn.Sequential` chains layers in order. The output of one layer becomes the
input to the next. This provides operations and components useful in building a standard feedforward neural network.

In [ ]:
# A two-hidden-layer network: 784 → 128 → 64 → 10
model = nn.Sequential(
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 10),
)

# Run a forward pass with a fake batch of 8 "images" (28x28 = 784 pixels each)
# Inspired by MNIST data.
x = torch.randn(8, 784)
logits = model(x)
print("Input  shape:", x.shape)       # (8, 784)
print("Output shape:", logits.shape)  # (8, 10)

# Print the architecture
print("\nModel:\n", model)

---
## Writing a Custom `nn.Module`

For anything beyond a simple stack of layers you should create a subclass of `nn.Module`. This is done by defining your own "constructor" `__init__` (to register layers) and a function `forward` (to describe the computation). This is a common pattern used to define neural network architectures in a PyTorch model.

In [ ]:
class MLP(nn.Module):
    """Multi-layer perceptron with configurable hidden layers."""

    def __init__(self, in_dim, hidden_dims, out_dim):
        super().__init__()

        # Build a list of (Linear + ReLU) blocks dynamically
        layers = []
        prev_dim = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU())
            prev_dim = h
        layers.append(nn.Linear(prev_dim, out_dim))

        # Wrap in nn.Sequential so PyTorch tracks the parameters
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


model = MLP(in_dim=2, hidden_dims=[64, 64, 32], out_dim=1)
print(model)

x = torch.randn(10, 2)   # 10 points in 2-D
y = model(x)
print("\nOutput shape:", y.shape)  # (10, 1)

---
## Inspecting Parameters

The `model.parameters()` yields every learnable tensor in the model. Optimizers
use this to know what to update each step.

In [ ]:
total = 0
for name, param in model.named_parameters():
    n = param.numel()
    total += n
    print(f"{name:30s}  shape={tuple(param.shape)}  params={n}")

print(f"\nTotal trainable parameters: {total:,}")

---
## Loss Functions

A **loss function** is used to measure how wrong a model's predictions are.
During training we minimize the loss by adjusting the model's parameters (using an optimization method).

### Example: Mean Squared Error (MSE): a loss widely used in regression.

$$\mathcal{L} = \frac{1}{N}\sum_{i=1}^N (\hat{y}_i - y_i)^2$$

In [ ]:
mse = nn.MSELoss()

predictions = torch.tensor([1.0, 2.0, 3.0])
targets     = torch.tensor([1.1, 2.5, 2.8])

loss = mse(predictions, targets)
print(f"MSE loss: {loss.item():.4f}")

# Manually verify
manual = ((predictions - targets) ** 2).mean()
print(f"Manual  : {manual.item():.4f}")

### Example: Cross-Entropy Loss (classification)

This loss is widely used for classification tasks. It is related to using the method of maximum likelihood for selecting a model to decide how to assign a label $y$ to input $x$. This loss expects *raw logits* $z_j$ (not softmax) from the neural-network or other PyTorch models. The index $j$ is an integer for the class label $[1,2,\ldots,M]$. We use a probability $y_j^{[i]}$ for that the label should be $j$ for each sample input $x^{[i]}$. For $N$ samples $\{x^{[i]}\}{i=1}^N$, the cross-entropy loss is

$$\mathcal{L} = -\frac{1}{N}\sum_{i=1}^N \sum_{j=1}^M y_j^{[i]}\log\left(\frac{e^{z_{j}^{[i]}}}{\sum_k e^{z_{k}^{[i]}}}\right).$$

In [ ]:
ce = nn.CrossEntropyLoss()

# Batch of N=4 samples, M=3-class problem. Logits are raw (not softmaxed).
logits  = torch.tensor([[2.0, 1.0, 0.1],
                         [0.5, 2.5, 0.3],
                         [0.1, 0.2, 3.0],
                         [1.0, 1.0, 1.0]])
targets = torch.tensor([0, 1, 2, 0])   # true class indices

loss = ce(logits, targets)
print(f"Cross-entropy loss: {loss.item():.4f}")

# Predicted classes
preds = logits.argmax(dim=1)
accuracy = (preds == targets).float().mean()
print(f"Accuracy: {accuracy.item():.0%}")

---
## Putting It Together: One Forward + Loss Step

Here is the skeleton of a single training step (without the optimizer yet). We explain optimizers in the next notebook.

In [ ]:
torch.manual_seed(0)

# Toy regression problem: map 3 inputs → 1 output
model     = MLP(in_dim=3, hidden_dims=[16, 16], out_dim=1)
loss_fn   = nn.MSELoss()

# Fake data: 5 samples
x_batch   = torch.randn(5, 3)
y_batch   = torch.randn(5, 1)   # ground-truth targets

# Forward pass
y_pred    = model(x_batch)

# Compute loss
loss      = loss_fn(y_pred, y_batch)

print(f"Predictions :\n{y_pred.detach()}")
print(f"Targets     :\n{y_batch}")
print(f"Loss        : {loss.item():.4f}")

---
## Summary

| Concept | Code |
|---------|------|
| Linear layer | `nn.Linear(in, out)` |
| Common activations | `nn.ReLU()`, `nn.Tanh()`, `nn.GELU()` |
| Stack layers | `nn.Sequential(layer1, act1, layer2, …)` |
| Custom model | Subclass `nn.Module`, define `__init__` + `forward` |
| Count params | `sum(p.numel() for p in model.parameters())` |
| Regression loss | `nn.MSELoss()` |
| Classification loss | `nn.CrossEntropyLoss()` (takes raw logits) |

**Next:** [04_optimization.ipynb](04_optimization.ipynb). We explain next how to minimize the loss and update the model's parameters.